# The Privileged Access Hypothesis

**Survey Reference:** Section 6.3 - The Privileged Access Hypothesis

A model used to interpret itself can, under appropriate elicitation, access internal variables and intermediate computations that are not reliably recoverable from external behavioral analysis alone, providing a comparative advantage as an interpreter of its own behavior.


## Feature Descriptions Task

This notebook demonstrates the **Privileged Access Hypothesis** using the Feature Descriptions task from [Training Language Models To Explain Their Own Computations](https://arxiv.org/abs/2511.08579) (Li et al., 2025).

**Key Insight:** Models trained to describe their own internal features (SAE features from their residual stream) can produce better explanations than external models, suggesting they have privileged access to their own computations.

The Feature Descriptions task demonstrates **privileged access** by training models to:
1. Take a continuous feature vector from the target model's residual stream
2. Generate a natural language description of what that feature represents

### Models:
| Model | Description |
|-------|-------------|
| `Transluce/features_explain_llama3.1_8b_llama3.1_8b` | Llama-3.1-8B explains **itself** |
| `Transluce/features_explain_llama3.1_8b_llama3.1_8b_instruct` | Llama-3.1-8B-Instruct explains Llama-3.1-8B |
| `Transluce/features_explain_llama3.1_8b_llama3_8b` | Llama-3-8B explains Llama-3.1-8B |


In [ ]:
from model.utils import load_models
from transformers import AutoTokenizer
import yaml

config_path = "model/base_131k.yaml"
target_model_path = "meta-llama/Llama-3.1-8B"

feature_explainer_models = {
    "Llama-3.1-8B": "Transluce/features_explain_llama3.1_8b_llama3.1_8b",
    "Llama-3.1-8B-Instruction": "Transluce/features_explain_llama3.1_8b_llama3.1_8b_instruct",
    "Llama-3-8B": "Transluce/features_explain_llama3.1_8b_llama3_8b",
}

with open(config_path, "r") as f:
    config = yaml.safe_load(f)


In [ ]:
# Load SAE features and explanations of the target model (Neuronpedia)
# We provide 3 features from layer-15 for demonstration

# To test all features, download the data by:
# SAE features + Neuronpedia explanations (training + in-distribution eval)
# wget https://transluce-public.s3.us-east-1.amazonaws.com/introspective-interp/SAE_feature_explanations_llama3.1_8b.tar.gz
# tar -xzvf SAE_feature_explanations_llama3.1_8b.tar.gz

import json
import pickle

print("Loading cached explanations and features...")
layer = 15
demo_explanations = json.load(open(f"./demo_data/three_{layer}layer_sae_feature_descriptions.json", "r"))
demo_features = pickle.load(open(f"./demo_data/three_{layer}layer_sae_features.pkl", "rb"))
demo_feature_ids = [dee[0][1] for dee in demo_explanations]

In [ ]:
# The templates used to prompt the feature explainer model
# three special tokens are used to denote the begin and end of continuous features, and the feature placeholder

special_tokens = config["continuous_tokens"]

prompt_args = {
    "layer": str(layer),
    "begin_continuous": special_tokens["begin_continuous"],
    "end_continuous": special_tokens["end_continuous"],
    "feature": special_tokens["continuous_rep"],
}
templates = [
    "We can describe {begin_continuous}{feature}{end_continuous} at layer {layer} as encoding ",
    "Generate a description of this feature at layer {layer}: {begin_continuous}{feature}{end_continuous}.",
    "What does {begin_continuous}{feature}{end_continuous} encode at layer {layer}?",
]
input_with_placeholder = [template.format(**prompt_args) for template in templates]

In [ ]:
import torch


@torch.no_grad()
def generate_description(model, tokenizer, input_example, target_feature):
    model.eval()
    if "ContinuousQwen3ForCausalLM" in model.config.architectures:
        stop_strings = None
    else:
        stop_strings = [" [END]", "\n", ">>>."]

    tokenized_inputs = tokenizer(
        [input_example],
        padding=False,
        truncation=True,
        return_tensors="pt",
        padding_side="left",
    )

    generation_outputs = model.generate(
        input_ids=tokenized_inputs["input_ids"].to(model.device),
        attention_mask=tokenized_inputs["attention_mask"].to(model.device),
        max_new_tokens=30,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        do_sample=False,
        temperature=None,
        top_p=None,
        inputs_continuous_tokens=[target_feature],
        return_dict_in_generate=True,
        output_hidden_states=True,
        stop_strings=stop_strings,
        tokenizer=tokenizer,
        extra_args=[],
    )

    prediction = tokenizer.decode(
        generation_outputs["sequences"][0][len(tokenized_inputs["input_ids"][0]) :],
        skip_special_tokens=True,
    ).strip()
    return prediction

In [ ]:

results = {feature_id: {"label": demo_explanations[i][1]["chosen_label"]} for i, feature_id in enumerate(demo_feature_ids)}

for explainer_model_name, explainer_model_path in feature_explainer_models.items():
    print(f"\n=== Using {explainer_model_name} (Explainer) to explain the features of {target_model_path} (Target) ===")


    # Load tokenizer
    model_tokenizer = AutoTokenizer.from_pretrained(config["model_path"], padding_side="left")
    model_tokenizer.pad_token = model_tokenizer.eos_token
    model_tokenizer.pad_token_id = model_tokenizer.eos_token_id

    target_tokenizer = AutoTokenizer.from_pretrained(target_model_path, padding_side="left")
    target_tokenizer.pad_token = target_tokenizer.eos_token
    target_tokenizer.pad_token_id = target_tokenizer.eos_token_id

    special_token_ids = {
        cont_token: model_tokenizer.convert_tokens_to_ids(config["continuous_tokens"][cont_token])
        for cont_token in config["continuous_tokens"]
    }

    # load model
    model, target_model, _ = load_models(
        predictor_model_path=explainer_model_path,
        target_model_path=target_model_path,
        special_tokens_ids=special_token_ids,
        cache_dir=None,
        train_self=False,
        use_bf16=config["train"].get("bf16", True),
        batch_size=config["test"]["batch_size"],
        ckpt_dir=os.path.dirname(explainer_model_path.rstrip("/")),
        use_embed_proj=config.get("use_embed_proj", False),
        embed_proj_path=config.get("embed_proj_path", None),
        predictor_model_type=config["model_path"],
    )

    # generate descriptions for the SAE features
    for i in range(len(demo_features)):
        prediction = generate_description(
            model,
            model_tokenizer,
            input_with_placeholder[i],
            demo_features[i],
        )
        # print(f"\nFeature {i+1} Description Generated by {explainer_model_name}:")
        # print(f"Prompt: {input_with_placeholder[i]}")
        # print(f"Prediction: {prediction}")
        # print(f"Label (Neuronpedia): {demo_explanations[i][1]['chosen_label']}")

        results[demo_feature_ids[i]][explainer_model_name] = prediction

    del model, target_model
    torch.cuda.empty_cache()

print(json.dumps(results, indent=2))

## References

1. Li, B. Z., Guo, Z. C., Huang, V., Steinhardt, J., & Andreas, J. (2025). *Training Language Models to Explain Their Own Computations*. arXiv:2511.08579.

---

**Next:** → **04_universality_platonic_representation.ipynb**: Do different models learn the same representations?